# Byte-Gemma retrain (same dataset, fixed training setup)
Runtime → Change runtime type → T4 GPU. Upload `train.jsonl` to the Colab files panel before running cell 4.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it",
    max_seq_length=2048,
    load_in_4bit=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# FIX 1: apply Gemma's chat template so every assistant reply ends with <end_of_turn>
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma3")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")

def fmt(batch):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
        for m in batch["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(fmt, batched=True, remove_columns=dataset.column_names)
print(dataset[0]["text"])  # check: must contain <start_of_turn>model ... <end_of_turn>

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# FIX 2: compute loss only on the tutor's replies, never on system/user text
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

In [ ]:
# Sanity check: masked (user/system) parts show as blanks, only tutor replies remain
print(tokenizer.decode([t if t != -100 else tokenizer.pad_token_id for t in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " "))

In [ ]:
trainer.train()

In [ ]:
# Test: replies must STOP by themselves
FastLanguageModel.for_inference(model)
SYSTEM = open("prompt.txt").read() if __import__("os").path.exists("prompt.txt") else "You are Byte, a patient C++ tutor for beginners."
msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":"What is the difference between struct and class in C++?"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=False))

In [ ]:
# Export GGUF (Q4_K_M, same as before)
model.save_pretrained_gguf("byte-gemma", tokenizer, quantization_method="q4_k_m")

# Optional: push to Hugging Face
# model.push_to_hub_gguf("99Amit99/byte-gemma-3-1b-it", tokenizer, quantization_method="q4_k_m", token="hf_xxx")